In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
df_csv = pd.read_csv('data.csv')
df_excel = pd.read_excel('data.xlsx')

In [ ]:
df = df_csv.copy()

In [ ]:
# Encode categorical features using LabelEncoder
# Note: LabelEncoder assigns numeric values alphabetically
label_encoders = {}
for col in ["Cut", "Color", "Clarity", "Polish", "Symmetry", "Report"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [ ]:
# Separate features and target variable
X = df.drop(columns=["ID", "Price"])
y = df["Price"]


In [ ]:
# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Scale features — fit on train only to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)


RandomForestRegressor(random_state=42)

In [ ]:
# Evaluate model performance
y_pred = model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae}, R2: {r2}")

MAE: 669.1049026846069, R2: 0.9850470156109193


In [ ]:
# Apply model to full dataset to get predicted prices for all diamonds
df_scaled = scaler.transform(X)
df["Predicted Price"] = model.predict(df_scaled)


In [ ]:
# Filter undervalued diamonds: quality cut/color/clarity, actual price below predicted
best_diamonds = df[(df["Cut"] >= label_encoders["Cut"].transform(["Very Good"])[0]) &
                   (df["Color"] <= label_encoders["Color"].transform(["H"])[0]) &
                   (df["Clarity"] <= label_encoders["Clarity"].transform(["VS1"])[0]) &
                   (df["Price"] < df["Predicted Price"])]

best_diamonds["Discount"] = best_diamonds["Predicted Price"] - best_diamonds["Price"]
top_diamonds = best_diamonds.nlargest(5, "Discount")


# Sort by largest discount and take top 5
top_diamonds = best_diamonds.sort_values("Discount", ascending=False).head(5)

/var/folders/r2/wft342rn5h98dhjs_dbmn05h0000gn/T/ipykernel_18998/3466456747.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  best_diamonds["Discount"] = best_diamonds["Predicted Price"] - best_diamonds["Price"]


In [ ]:
# Decode labels back to original category names
for col in label_encoders:
    top_diamonds[col] = label_encoders[col].inverse_transform(top_diamonds[col])


In [ ]:
# Display top 5 undervalued diamonds
print(top_diamonds[["ID", "Carat Weight", "Cut", "Color", "Clarity", "Price", "Predicted Price", "Discount"]])


        ID  Carat Weight        Cut Color Clarity  Price  Predicted Price  \
2995  2996          1.91  Very Good     F      IF  28557     41885.160000   
5563  5564          2.61  Very Good     E      IF  67240     75643.090000   
5559  5560          1.84  Very Good     E      IF  35410     41511.435000   
3256  3257          2.59  Very Good     E     SI1  25158     30955.820000   
742    743          2.01  Very Good     G     VS1  21524     25486.963119   

          Discount  
2995  13328.160000  
5563   8403.090000  
5559   6101.435000  
3256   5797.820000  
742    3962.963119  
